# Proyecto Final | Computación Flexible

Mantenimiento predictivo para Equipos Industriales con el dataset **Predictive Maintenance AI4I 2020 (UCI)**.

## Objetivo

El objetivo de este proyecto es aplicar lo aprendido a lo largo del curso, en especifico: integrar redes neuronales, sistemas difusos y algoritmos genéticos, cuidando interpretabilidad y evitando la fuga de datos.

## 0. Setup y reproducibilidad

In [1]:
import numpy as np
import pandas as pd


In [2]:
SEED = 42
np.random.seed(SEED)

## 1. Planteamiento

## 2. Carga y descripción del dataset

In [3]:
df = pd.read_csv("./dataset/ai4i2020.csv")

print(df.shape)
df.info()

(10000, 14)
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9

## 3. EDA

### 3.1 Tipos, nulos, duplicados, consistencia de etiquetas

In [4]:
print("Nulos totales:", df.isna().sum().sum())
print("Duplicados:", df.duplicated().sum())

Nulos totales: 0
Duplicados: 0


In [5]:
modos = ["TWF", "HDF", "PWF", "OSF", "RNF"]

algun_modo = df[modos].any(axis=1).astype(int)
print(pd.crosstab(df["Machine failure"], algun_modo,
                  rownames=["Machine failure"], colnames=["Algún modo"]))

Algún modo          0    1
Machine failure           
0                9643   18
1                   9  330


In [6]:
cols_num = ["Type", "Air temperature [K]", "Process temperature [K]",
            "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]

# Caso A: modo activo, pero sin falla
caso_a = df[(df["Machine failure"] == 0) & algun_modo.astype(bool)]
print("Caso A:", len(caso_a))
print(caso_a[modos].sum())

# Caso B: falla sin modo registrado
caso_b = df[(df["Machine failure"] == 1) & ~algun_modo.astype(bool)]
print("\nCaso B:", len(caso_b))
print(caso_b[cols_num])

# Filas con más de un modo simultáneo
print("\nFilas multi-modo:", (df[modos].sum(axis=1) > 1).sum())

Caso A: 18
TWF     0
HDF     0
PWF     0
OSF     0
RNF    18
dtype: int64

Caso B: 9
     Type  Air temperature [K]  Process temperature [K]  \
1437    H                298.8                    309.9   
2749    M                299.7                    309.2   
4044    M                301.9                    310.9   
4684    M                303.6                    311.8   
5536    M                302.3                    311.8   
5941    L                300.6                    310.7   
6478    L                300.5                    309.8   
8506    L                298.4                    309.6   
9015    L                297.2                    308.1   

      Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  
1437                    1439         45.2               40  
2749                    1685         28.9              179  
4044                    1419         47.7               20  
4684                    1421         44.8              101  
5536               

In [7]:
# Limpieza: eliminar fallas sin causa registrada (caso B)
df_clean = df.drop(index=caso_b.index).reset_index(drop=True)
print(df_clean.shape)
print(df_clean["Machine failure"].value_counts())

(9991, 14)
Machine failure
0    9661
1     330
Name: count, dtype: int64


### 3.2 Estadísticas descriptivas

In [8]:
features = ["Air temperature [K]", "Process temperature [K]",
            "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]

# Descriptivas globales
display(df_clean[features].describe().T)

# Descriptivas por clase
display(df_clean.groupby("Machine failure")[features].agg(["mean", "std"]).T)

,count,mean,std,min,25%,50%,75%,max
Air temperature [K],9991.0,300.004634,2.000300,295.3,298.3,300.1,301.5,304.5
Process temperature [K],9991.0,310.005385,1.483990,305.7,308.8,310.1,311.1,313.8
Rotational speed [rpm],9991.0,1538.804124,179.321027,1168.0,1423.0,1503.0,1612.0,2886.0
Torque [Nm],9991.0,39.985377,9.969029,3.8,33.2,40.1,46.7,76.6
Tool wear [min],9991.0,107.942648,63.656648,0.0,53.0,108.0,162.0,253.0


Machine failure                         0            1
Air temperature [K]     mean   299.973999   300.901515
                        std      1.990748     2.073226
Process temperature [K] mean   309.995570   310.292727
                        std      1.486846     1.369004
Rotational speed [rpm]  mean  1540.260014  1496.181818
                        std    167.394734   389.590780
Torque [Nm]             mean    39.629655    50.399394
                        std      9.472080    16.457729
Tool wear [min]         mean   106.693717   144.506061
                        std     62.945790    72.939549

### 3.3 Distribución de clases y modos de falla

In [9]:
print(df["Machine failure"].value_counts(normalize=True))
print(df[modos].sum())

Machine failure
0    0.9661
1    0.0339
Name: proportion, dtype: float64
TWF     46
HDF    115
PWF     95
OSF     98
RNF     19
dtype: int64


In [10]:
# Distribución de Type y tasa de falla por tipo
print(df_clean["Type"].value_counts(normalize=True))
print(df_clean.groupby("Type")["Machine failure"].mean())

Type
L    0.60014
M    0.29957
H    0.10029
Name: proportion, dtype: float64
Type
H    0.019960
L    0.038526
M    0.026395
Name: Machine failure, dtype: float64


### 3.4 Distribuciones por clase (falla vs no falla)

### 3.5 Correlaciones

### 3.6 Verificación de las reglas físicas

## 4. Exploración no supervisada (comparativa)

### 4.1 Codo + silueta → k para K-Means

### 4.2 K-Means vs DBSCAN vs HDBSCAN

### 4.3 ¿Los clusters capturan regímenes de falla? (tasa de falla por cluster)


## 5. Preprocesamiento


### 5.1 Eliminación de IDs y columnas con fuga


### 5.2 Codificación de Type


### 5.3 Ingeniería de características


### 5.4 Split estratificado → escalado/SMOTE solo en train (Pipeline)


### 5.5 Selección de características


## 6. Modelo base


### 6.1 Regresión logística + Árbol/Random Forest


### 6.2 CV estratificada, métricas, matriz de confusión, curvas PR/ROC


### 6.3 Interpretabilidad del baseline (coeficientes / importancia)